# Data Augmentation: Images, Text, and Tabular

Augmentation is a form of regularization: it increases effective training set size by creating
modified versions of existing samples. The model learns invariances from the augmentations.

**Why it works (regularization view):** Augmentation makes the training distribution broader,
preventing the model from memorizing specific pixel patterns / word orderings. It implicitly
encodes prior knowledge ("a flipped cat is still a cat").

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Image Augmentation

We implement common augmentations from scratch on numpy arrays (H, W, C format).

In [ ]:
# Create a synthetic "image" (64x64 RGB with patterns)
def create_test_image(size=64):
    """Create a test image with identifiable patterns."""
    img = np.zeros((size, size, 3), dtype=np.float32)
    # Gradient background
    for i in range(size):
        img[i, :, 0] = i / size  # Red gradient top-to-bottom
        img[:, i, 2] = i / size  # Blue gradient left-to-right
    # Add a rectangle
    img[10:30, 15:45, 1] = 0.8
    # Add a circle
    y, x = np.ogrid[-32:32, -32:32]
    mask = x**2 + y**2 <= 10**2
    img[mask, :] = [1.0, 1.0, 0.0]  # Yellow circle in center
    return np.clip(img, 0, 1)

img = create_test_image(64)
plt.figure(figsize=(3, 3))
plt.imshow(img)
plt.title('Original')
plt.axis('off')
plt.show()

In [ ]:
def random_horizontal_flip(img, p=0.5):
    """Flip image horizontally with probability p."""
    if np.random.random() < p:
        return img[:, ::-1, :].copy()
    return img.copy()

def random_vertical_flip(img, p=0.5):
    """Flip image vertically with probability p."""
    if np.random.random() < p:
        return img[::-1, :, :].copy()
    return img.copy()

def random_rotation_90(img):
    """Rotate image by a random multiple of 90 degrees."""
    k = np.random.randint(0, 4)  # 0, 90, 180, 270
    return np.rot90(img, k, axes=(0, 1)).copy()

def random_crop(img, crop_size):
    """Random crop of given size from the image."""
    h, w = img.shape[:2]
    ch, cw = crop_size
    if ch >= h or cw >= w:
        return img.copy()
    top = np.random.randint(0, h - ch)
    left = np.random.randint(0, w - cw)
    return img[top:top+ch, left:left+cw, :].copy()

def color_jitter(img, brightness=0.2, contrast=0.2, saturation=0.2):
    """Random color jittering."""
    result = img.copy()
    # Brightness
    result = result + np.random.uniform(-brightness, brightness)
    # Contrast
    mean = result.mean()
    factor = np.random.uniform(1 - contrast, 1 + contrast)
    result = (result - mean) * factor + mean
    # Simple saturation adjustment
    gray = result.mean(axis=2, keepdims=True)
    sat_factor = np.random.uniform(1 - saturation, 1 + saturation)
    result = gray + (result - gray) * sat_factor
    return np.clip(result, 0, 1)

def cutout(img, n_holes=1, hole_size=16):
    """Cutout: randomly mask out square regions."""
    result = img.copy()
    h, w = img.shape[:2]
    for _ in range(n_holes):
        y = np.random.randint(0, h)
        x = np.random.randint(0, w)
        y1 = max(0, y - hole_size // 2)
        y2 = min(h, y + hole_size // 2)
        x1 = max(0, x - hole_size // 2)
        x2 = min(w, x + hole_size // 2)
        result[y1:y2, x1:x2, :] = 0  # Black patch
    return result

# Visualize all augmentations
augmentations = [
    ('Original', img),
    ('H-Flip', random_horizontal_flip(img, p=1.0)),
    ('V-Flip', random_vertical_flip(img, p=1.0)),
    ('Rotation 90', random_rotation_90(img)),
    ('Random Crop', random_crop(img, (48, 48))),
    ('Color Jitter', color_jitter(img, 0.3, 0.3, 0.3)),
    ('Cutout', cutout(img, n_holes=2, hole_size=16)),
]

fig, axes = plt.subplots(1, len(augmentations), figsize=(20, 3))
for ax, (name, aug_img) in zip(axes, augmentations):
    ax.imshow(aug_img)
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.suptitle('Image Augmentation Gallery', fontsize=14)
plt.tight_layout()
plt.show()

## Mixup

Mixup creates virtual training examples by linearly interpolating between pairs of samples:

$$\tilde{x} = \lambda x_i + (1 - \lambda) x_j$$
$$\tilde{y} = \lambda y_i + (1 - \lambda) y_j$$

where $\lambda \sim \text{Beta}(\alpha, \alpha)$, typically $\alpha \in [0.1, 0.4]$.

**Why it works:** Regularizes by encouraging linear behavior between training examples. Reduces overconfidence and improves calibration.

In [ ]:
def mixup(images, labels, alpha=0.2):
    """Mixup augmentation for a batch of images and labels.
    
    images: (batch_size, H, W, C)
    labels: (batch_size,) or one-hot (batch_size, n_classes)
    """
    batch_size = len(images)
    lam = np.random.beta(alpha, alpha, batch_size)
    
    # Random permutation for mixing partners
    perm = np.random.permutation(batch_size)
    
    mixed_images = np.zeros_like(images)
    mixed_labels = np.zeros_like(labels, dtype=float)
    
    for i in range(batch_size):
        l = lam[i]
        mixed_images[i] = l * images[i] + (1 - l) * images[perm[i]]
        mixed_labels[i] = l * labels[i] + (1 - l) * labels[perm[i]]
    
    return mixed_images, mixed_labels, lam

# Create two distinct images
img1 = create_test_image(64)
img2 = np.zeros((64, 64, 3), dtype=np.float32)
img2[20:44, 20:44, :] = [0, 0.7, 1.0]  # Blue square
img2[:, :, 0] = 0.3  # Red tint

# Show mixup at different lambda values
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, lam in zip(axes, [1.0, 0.75, 0.5, 0.25, 0.0]):
    mixed = lam * img1 + (1 - lam) * img2
    ax.imshow(np.clip(mixed, 0, 1))
    ax.set_title(f'lambda={lam:.2f}\ny={lam:.2f}*cat + {1-lam:.2f}*dog')
    ax.axis('off')
plt.suptitle('Mixup at Different Lambda Values', fontsize=14)
plt.tight_layout()
plt.show()

## CutMix

CutMix combines images by cutting a patch from one and pasting it onto another:

$$\tilde{x} = M \odot x_i + (1 - M) \odot x_j$$
$$\tilde{y} = \lambda y_i + (1 - \lambda) y_j$$

where $M$ is a binary mask and $\lambda$ is the area ratio of the patch.

**Advantage over Mixup:** preserves local image structure; no blurring.

In [ ]:
def cutmix(img1, img2, label1, label2, alpha=1.0):
    """CutMix: cut and paste rectangular patch."""
    h, w = img1.shape[:2]
    lam = np.random.beta(alpha, alpha)
    
    # Sample bounding box
    cut_ratio = np.sqrt(1 - lam)
    cut_h = int(h * cut_ratio)
    cut_w = int(w * cut_ratio)
    
    cy = np.random.randint(0, h)
    cx = np.random.randint(0, w)
    
    y1 = max(0, cy - cut_h // 2)
    y2 = min(h, cy + cut_h // 2)
    x1 = max(0, cx - cut_w // 2)
    x2 = min(w, cx + cut_w // 2)
    
    # Actual lambda based on clipped box
    actual_lam = 1 - (y2 - y1) * (x2 - x1) / (h * w)
    
    result = img1.copy()
    result[y1:y2, x1:x2, :] = img2[y1:y2, x1:x2, :]
    
    mixed_label = actual_lam * label1 + (1 - actual_lam) * label2
    
    return result, mixed_label, actual_lam

# Demo
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(img1); axes[0].set_title('Image A'); axes[0].axis('off')
axes[1].imshow(np.clip(img2, 0, 1)); axes[1].set_title('Image B'); axes[1].axis('off')

for i, ax in enumerate(axes[2:]):
    result, label, lam = cutmix(img1, img2, 1.0, 0.0)
    ax.imshow(np.clip(result, 0, 1))
    ax.set_title(f'CutMix (lam={lam:.2f})')
    ax.axis('off')

plt.suptitle('CutMix Examples', fontsize=14)
plt.tight_layout()
plt.show()

## Text Augmentation

Text augmentation is trickier than images because small changes can alter meaning.

In [ ]:
# Simple synonym dictionary
SYNONYMS = {
    'good': ['great', 'excellent', 'fine', 'nice'],
    'bad': ['poor', 'terrible', 'awful', 'horrible'],
    'happy': ['glad', 'joyful', 'pleased', 'delighted'],
    'sad': ['unhappy', 'sorrowful', 'gloomy', 'melancholy'],
    'big': ['large', 'huge', 'enormous', 'massive'],
    'small': ['tiny', 'little', 'miniature', 'compact'],
    'fast': ['quick', 'rapid', 'swift', 'speedy'],
    'movie': ['film', 'picture', 'motion picture'],
    'like': ['enjoy', 'appreciate', 'love'],
}

def synonym_replacement(text, n=1):
    """Replace n random words with synonyms."""
    words = text.split()
    new_words = words.copy()
    replaceable = [(i, w.lower()) for i, w in enumerate(words) if w.lower() in SYNONYMS]
    
    if not replaceable:
        return text
    
    n_replace = min(n, len(replaceable))
    chosen = [replaceable[i] for i in np.random.choice(len(replaceable), n_replace, replace=False)]
    
    for idx, word in chosen:
        synonym = np.random.choice(SYNONYMS[word])
        new_words[idx] = synonym
    
    return ' '.join(new_words)

def random_insertion(text, n=1):
    """Insert a random synonym of a random word at a random position."""
    words = text.split()
    for _ in range(n):
        replaceable = [w.lower() for w in words if w.lower() in SYNONYMS]
        if not replaceable:
            break
        word = np.random.choice(replaceable)
        synonym = np.random.choice(SYNONYMS[word])
        pos = np.random.randint(0, len(words) + 1)
        words.insert(pos, synonym)
    return ' '.join(words)

def random_deletion(text, p=0.1):
    """Randomly delete words with probability p."""
    words = text.split()
    if len(words) <= 1:
        return text
    remaining = [w for w in words if np.random.random() > p]
    if not remaining:
        return words[np.random.randint(len(words))]  # Keep at least one
    return ' '.join(remaining)

def random_swap(text, n=1):
    """Randomly swap two words n times."""
    words = text.split()
    new_words = words.copy()
    for _ in range(n):
        if len(new_words) < 2:
            break
        i, j = np.random.choice(len(new_words), 2, replace=False)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return ' '.join(new_words)

# Demo
text = "I really like this good movie it was a fast and happy experience"
print(f"Original:     {text}")
print(f"Synonym:      {synonym_replacement(text, n=2)}")
print(f"Insertion:    {random_insertion(text, n=1)}")
print(f"Deletion:     {random_deletion(text, p=0.2)}")
print(f"Swap:         {random_swap(text, n=2)}")

### Back-Translation (Concept)

Translate text to another language and back. This paraphrases while preserving meaning.

```
English -> French -> English
"The cat sat on the mat" -> "Le chat etait assis sur le tapis" -> "The cat was sitting on the carpet"
```

Requires a translation API/model but produces high-quality augmentations for NLP.

## Tabular Data Augmentation

Augmenting tabular data is less studied. Main approaches:
1. **SMOTE** (see notebook 04) -- interpolation between neighbors
2. **Gaussian noise injection** -- add small noise to continuous features
3. **Feature dropout** -- randomly zero out features during training

In [ ]:
def noise_injection(X, noise_std=0.1):
    """Add Gaussian noise to features."""
    noise = np.random.randn(*X.shape) * noise_std
    return X + noise

def feature_dropout(X, drop_prob=0.1):
    """Randomly zero out features."""
    mask = np.random.random(X.shape) > drop_prob
    return X * mask

# Demo
X_demo = np.random.randn(5, 4)
print("Original:")
print(np.round(X_demo, 3))
print("\nWith noise (std=0.1):")
print(np.round(noise_injection(X_demo, 0.1), 3))
print("\nWith feature dropout (p=0.3):")
print(np.round(feature_dropout(X_demo, 0.3), 3))

## Visualize Multiple Augmented Versions of an Image

In [ ]:
def random_augment(img):
    """Apply a random combination of augmentations."""
    result = img.copy()
    if np.random.random() < 0.5:
        result = random_horizontal_flip(result, p=1.0)
    if np.random.random() < 0.3:
        result = random_vertical_flip(result, p=1.0)
    if np.random.random() < 0.5:
        result = color_jitter(result, 0.2, 0.2, 0.2)
    if np.random.random() < 0.3:
        result = cutout(result, n_holes=1, hole_size=12)
    return result

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0, 0].imshow(img)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for ax in axes.flat[1:]:
    aug = random_augment(img)
    ax.imshow(np.clip(aug, 0, 1))
    ax.set_title('Augmented')
    ax.axis('off')

plt.suptitle('10 Augmented Versions of the Same Image', fontsize=14)
plt.tight_layout()
plt.show()

## Notes: RandAugment and AutoAugment

**AutoAugment (2018):** Uses reinforcement learning to search for the best augmentation policy.
- Searches over: which augmentation, what magnitude, what probability
- Expensive: requires training thousands of child models
- Found policies transfer well across datasets

**RandAugment (2020):** Simpler alternative that works almost as well.
- Just two hyperparameters: N (number of augmentations to apply) and M (magnitude)
- Randomly pick N augmentations from a fixed set, each at magnitude M
- No search needed, easy to tune

```python
# RandAugment pseudocode
def rand_augment(image, N=2, M=9):
    ops = [rotate, shear, translate, brightness, contrast, ...]
    for _ in range(N):
        op = random.choice(ops)
        image = op(image, magnitude=M)
    return image
```

## Interview Questions

### Q: Why does augmentation work (regularization view)?
**A:** Augmentation artificially increases the diversity of the training set without collecting new data. It encodes prior knowledge about invariances (e.g., a flipped image has the same label). This prevents overfitting because the model can't memorize exact training patterns -- it must learn features that are robust to augmentations. Mathematically, it smooths the training distribution.

### Q: Mixup -- how and why?
**A:** Mixup creates virtual training examples by linear interpolation: $\tilde{x} = \lambda x_i + (1-\lambda) x_j$ with $\lambda \sim \text{Beta}(\alpha, \alpha)$. Labels are mixed the same way. This encourages the model to produce linear predictions between training examples, which:
1. Reduces overconfidence (better calibration)
2. Smooths the decision boundary
3. Acts as strong regularization

### Q: Augmentation in NLP -- what works?
**A:** NLP augmentation is harder because text is discrete and semantic:
- **Synonym replacement:** Safe but limited vocabulary coverage
- **Back-translation:** High quality paraphrasing, best general-purpose method
- **Random deletion/swap/insertion:** Noisy but effective with small datasets
- **LLM-based augmentation:** Use GPT/similar to generate paraphrases (modern approach)
- **Masking + filling:** Mask tokens and use a language model to fill alternatives

For NLP, augmentation matters most in low-resource settings. With enough data, pretrained models handle invariances through pretraining.

### Q: When NOT to augment?
**A:** When augmentations violate label invariance (e.g., flipping digits 6/9, rotating satellite images where orientation matters). Always think about whether the transformation preserves the label.